# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
import pandas as pd
import numpy as np

In [8]:
CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

In [17]:
# 1
def clean(path):
    df = pd.read_csv(path)
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print(f"суммы в числа {df.shape}")
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print(f"Города через словарь {df.shape}")
    df = df.drop_duplicates()
    print(f"Убрал дубликаты {df.shape}")
    df["order_date"] = pd.to_datetime(df["order_date"])
    print(f"Проставили формат дат {df.shape}")
    df = df[df["customer_id"] != "test"]
    print(f"Убрал тест {df.shape}")
    return df


In [18]:
clean("orders.csv")

суммы в числа (5463, 8)
Города через словарь (5463, 8)
Убрал дубликаты (5283, 8)
Проставили формат дат (5283, 8)
Убрал тест (5283, 8)


,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00:00,C00557,Астана,Электроника,22150.0,NaN,оплачен
5461,103460,2026-10-31 22:00:00,C02162,Алматы,Электроника,24670.0,2.0,отменён


In [7]:
df = pd.read_csv("orders.csv")
len(df)

5463

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [29]:
# 2

df1 = clean("orders.csv")
sep = df1["order_date"] < "2026-10"
sep

суммы в числа (5463, 8)
Города через словарь (5463, 8)
Убрал дубликаты (5283, 8)
Проставили формат дат (5283, 8)
Убрал тест (5283, 8)


0        True
1        True
2        True
3        True
4        True
        ...  
5458    False
5459    False
5460    False
5461    False
5462    False
Name: order_date, Length: 5283, dtype: bool

In [30]:
octo = ~sep
octo

0       False
1       False
2       False
3       False
4       False
        ...  
5458     True
5459     True
5460     True
5461     True
5462     True
Name: order_date, Length: 5283, dtype: bool

In [31]:
paid = df1["status"] == "оплачен"
paid

0        True
1        True
2        True
3        True
4        True
        ...  
5458     True
5459     True
5460     True
5461    False
5462     True
Name: status, Length: 5283, dtype: bool

In [33]:
df1.loc[paid & sep, "amount"].sum()

np.float64(55243390.0)

In [34]:
s, o = df1.loc[paid & sep, "amount"].sum(), df1.loc[paid & octo, "amount"].sum()
print(s, o, f"{o / s - 1:.1%}")

55243390.0 44419510.0 -19.6%


In [39]:
for city in df1["city"].unique():
    city_mask = df1["city"] == city

    s = df1.loc[paid & sep & city_mask, "amount"].sum()
    o = df1.loc[paid & octo & city_mask, "amount"].sum()

    change = (o / s - 1) * 100 

    print(f"{city}: сентябрь = {s}, октябрь = {o}, изменение = {change:.1f}%")

Алматы: сентябрь = 34501050.0, октябрь = 24643010.0, изменение = -28.6%
Астана: сентябрь = 9034220.0, октябрь = 8850530.0, изменение = -2.0%
Караганда: сентябрь = 3648520.0, октябрь = 3371680.0, изменение = -7.6%
Шымкент: сентябрь = 4558650.0, октябрь = 4044010.0, изменение = -11.3%
Атырау: сентябрь = 923990.0, октябрь = 969860.0, изменение = 5.0%
Актобе: сентябрь = 2576960.0, октябрь = 2540420.0, изменение = -1.4%


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [45]:
# 3
# Искажает данные TEST клиент. Изза того, что суммы большие и очень большой разброс

df2 = pd.read_csv("orders.csv")
df2["amount"] = pd.to_numeric(df2["amount"].str.replace(" ", ""))

sep = df2["order_date"] < "2026-10"
octo = ~sep
paid = df2["status"] == "оплачен"
s = df2.loc[paid & sep, "amount"].sum()
o = df2.loc[paid & octo, "amount"].sum()

print(f"Все данные  {o / s - 1:.1%}")

df2 = df2.drop_duplicates()
sep = df2["order_date"] < "2026-10"
octo = ~sep
paid = df2["status"] == "оплачен"
s = df2.loc[paid & sep, "amount"].sum()
o = df2.loc[paid & octo, "amount"].sum()

print(f"После удаления дубликатов:  {o / s - 1:.1%}")

df2 = df2[df2["customer_id"] != "TEST"]
sep = df2["order_date"] < "2026-10"
octo = ~sep
paid = df2["status"] == "оплачен"
s = df2.loc[paid & sep, "amount"].sum()
o = df2.loc[paid & octo, "amount"].sum()

print(f"После удаления тестовых данных: {o / s - 1:.1%}")



Все данные  -22.7%
После удаления дубликатов:  -19.6%
После удаления тестовых данных: -2.3%


## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [52]:
# 4
df1["delivery_days"].isna().sum()


np.int64(158)

In [64]:
df1.loc[df1["delivery_days"].isna(), ["city", "status"]].value_counts()

city       status 
Алматы     оплачен    61
Астана     оплачен    35
Караганда  оплачен    23
Шымкент    оплачен    17
Астана     отменён     5
Шымкент    отменён     5
Актобе     оплачен     4
Атырау     оплачен     4
Алматы     отменён     2
Атырау     отменён     1
Караганда  отменён     1
Name: count, dtype: int64

In [67]:
sh_df = df.loc[df["city"] == "Шымкент", "delivery_days"]
(sh_df.dropna() <=2).mean()

np.float64(0.30289855072463767)

In [70]:
sh_df.fillna(sh_df.median() <= 2).mean()

/var/folders/1p/0jqmy2zs1c7bv2vmd8qmyt7c0000gn/T/ipykernel_32758/1926492204.py:1: Pandas4Warning: 'bool' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  sh_df.fillna(sh_df.median() <= 2).mean()


np.float64(4.258790436005626)

In [71]:
(sh_df.fillna(0) <= 2).mean()

np.float64(0.3234880450070324)

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [88]:
# 5
df3 = pd.read_csv("orders.csv")
df3 = df3.drop_duplicates() # Не обновили датафрейм
df3["city"] = df3["city"].str.strip().str.lower().map(CITY) # у городов теперь 1 стандарт
print(df3["city"].nunique(), len(df3))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [91]:
pd.to_datetime(df1["order_date"])

0      2026-09-01 08:00:00
1      2026-09-01 08:00:00
2      2026-09-01 08:00:00
3      2026-09-01 08:00:00
4      2026-09-01 09:00:00
               ...        
5458   2026-10-31 21:00:00
5459   2026-10-31 21:00:00
5460   2026-10-31 22:00:00
5461   2026-10-31 22:00:00
5462   2026-10-31 22:00:00
Name: order_date, Length: 5283, dtype: datetime64[us]

In [99]:
# 6

df1["order_date"] = pd.to_datetime(df1["order_date"])
df1["month"] = df1["order_date"].dt.month
df1["day_name"] = df1["order_date"].dt.day_name()
df1["day_name"].value_counts()

day_name
Thursday     786
Tuesday      785
Friday       782
Saturday     775
Wednesday    774
Sunday       697
Monday       684
Name: count, dtype: int64

In [ ]:
max_day = df1["day_name"].value_counts().max()
len_days = len(df1)


5283

In [110]:
day_rate = df1["day_name"].value_counts() / len_days
print(f"Дни когда заказов меньше всего: {day_rate} ")


Дни когда заказов меньше всего: day_name
Thursday     0.148779
Tuesday      0.148590
Friday       0.148022
Saturday     0.146697
Wednesday    0.146508
Sunday       0.131933
Monday       0.129472
Name: count, dtype: float64 


In [111]:
df1["day_name"].value_counts() / max_day - 1

day_name
Thursday     0.000000
Tuesday     -0.001272
Friday      -0.005089
Saturday    -0.013995
Wednesday   -0.015267
Sunday      -0.113232
Monday      -0.129771
Name: count, dtype: float64

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8